In [1]:
import os
import numpy as np
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import chromadb

current_directory = Path.cwd()
path = f"{current_directory}\chroma_db"

client = chromadb.PersistentClient(path=path)

collection = client.get_collection('bbc_news')

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

hg_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

d:\AI Projects\RAG Practice\BBC_News_Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1245.68it/s]


In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6376.72it/s]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_context(query):
    query_embedding = hg_embeddings.embed_query(query)

    magnitude = np.linalg.norm(query_embedding)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "embeddings", "distances"]
    )
    print(results["distances"][0])

    # result_embeddings = results["embeddings"][0]
    # similarities = cosine_similarity(
    #     [query_embedding],
    #     result_embeddings
    # )[0]

    # print(similarities)

    print("\nFetched relevant documents:")
    print("---------------------------")
    documents = results["documents"][0]
    for doc in documents:
        print(doc,"\n")
        
    print("=================================================================================================================")

    pairs = [
        (query, document)
        for document in documents
    ]

    scores = reranker.predict(pairs)
    print(scores)

    scores_map = list(zip(scores, documents))
    scores_map.sort(reverse=True)

    ranked_docs = [doc for score, doc in scores_map[:3]]
    return ranked_docs

In [ ]:
def call_llm_with_rag(query):
    chunks = retrieve_context(query)
    context = "\n\n".join(chunks)

    print("\nReranked useful documents:")
    print("--------------------------")
    print(context)

    API_KEY = os.getenv('REQUESTY_API_KEY')

    client = OpenAI(
        base_url="https://router.requesty.ai/v1",
        api_key=API_KEY
    )
    
    system_prompt = f""""You are an expert summarizer and analyst.
Answer the user's question using only the information provided in the context.
Along with the response, provide supporting evidence specifying the documents that contain that information.
Do not introduce facts that are not supported by the context.
If the context is not sufficient to answer the question, just tell that without giving further explanation about other chunks.
"""

    user_prompt = f"""Context: {context}\n
Question: {query}
"""
 
    response = client.chat.completions.create(
        # model="nvidia/nemotron-3-ultra-550b-a55b",
        model="google/gemma-4-31b-it",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    answer = response.choices[0].message.content
    return answer

In [47]:
query = input()
answer = call_llm_with_rag(query)

[0.6054503321647644, 0.659854531288147, 0.6600468158721924, 0.6648604869842529, 0.6658828258514404]

Fetched relevant documents:
-----------------------------
tulu to appear at caledonian run twotime olympic metres champion derartu tulu has confirmed she will take part in the bupa great caledonian run in edinburgh on may the yearold ethiopian is the first star name to enter the event tulu has won the boston london and tokyo marathons as well as the world m title in we are delighted to have secured the services of one the most decorated competitors the sport has ever seen said race director matthew turnbull her record speaks for herself and there are few other women distance runners who would dare compare their pedigree with tulus he added she might be next month but that didnt stop her winning the olympic m bronze medal last summer shes an ultraconsistent championships racer 

chepkemei joins edinburgh lineup susan chepkemei has decided she is fit enough to run in next months great edi

In [48]:
print(answer)

if answer:
    content = f"""Qn: {query}
Ans:
{answer}

---------------------------------------------------------------------------------------------\n
"""

    with open('llm_responses.txt', 'a', encoding='utf-8') as f:
        f.write(content)

The provided context does not contain information regarding whether India has participated in any Olympics.
